In [2]:
!pip3 install "langchain" "langchain-community" "chromadb" "pypdf==4.3.1" "lark==1.1.9" 'posthog<6.0.0' langchain-huggingface huggingface-hub python-dotenv -U -q --break-system-packages

In [26]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

load_dotenv()


True

## Creating a retriever model

##### Creating the LLM

In [40]:
def llm():
    model_id = "mistralai/Mistral-Small-3.1-24B-Instruct-2503"

    parameters = {
        'temperature': 0.9,     # response creativity/randomness
        'top_p': 0.9,
    }

    token = os.getenv('HF_TOKEN') or os.getenv('HUGGINGFACEHUB_API_TOKEN')
    if not token:
        raise ValueError('Set HF_TOKEN or HUGGINGFACEHUB_API_TOKEN in your environment or .env file.')

    os.environ['HUGGINGFACEHUB_API_TOKEN'] = token

    endpoint = HuggingFaceEndpoint(
        repo_id=model_id,
        task='conversational',
        temperature=parameters['temperature'],
        top_p=parameters['top_p'],
    )

    return ChatHuggingFace(llm=endpoint)


##### Using Text Splitter

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

/opt/homebrew/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def text_splitter(data, chunk_size, chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)
    return chunks

##### Create the embedding model

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

def huggingface_embedding():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

### Vector Store-Backed Retriever

Wraps a vector store to match the retriever interface and retrieves documents using methods like similarity search or MMR.

MMR (Maximum Marginal Relevance) is a retrieval method that balances relevance to the query and diversity across results, thus avoiding returning many near-duplicate chunks by picking useful but non-redundant documents.

In [6]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/MZ9z1lm-Ui3YBp3SYWLTAQ/companypolicies.txt"

--2026-02-23 13:36:39--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/MZ9z1lm-Ui3YBp3SYWLTAQ/companypolicies.txt
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.45.118.108
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.45.118.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15660 (15K) [text/plain]
Saving to: ‘companypolicies.txt’

companypolicies.txt 100%[===================>]  15.29K  --.-KB/s    in 0.1s    

2026-02-23 13:36:40 (143 KB/s) - ‘companypolicies.txt’ saved [15660/15660]



In [8]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("companypolicies.txt")
txt_data = loader.load()
txt_data

[Document(metadata={'source': 'companypolicies.txt'}, page_content="1.\tCode of Conduct\n\nOur Code of Conduct outlines the fundamental principles and ethical standards that guide every member of our organization. We are committed to maintaining a workplace that is built on integrity, respect, and accountability.\nIntegrity: We hold ourselves to the highest ethical standards. This means acting honestly and transparently in all our interactions, whether with colleagues, clients, or the broader community. We respect and protect sensitive information, and we avoid conflicts of interest.\nRespect: We embrace diversity and value each individual's contributions. Discrimination, harassment, or any form of disrespectful behavior is unacceptable. We create an inclusive environment where differences are celebrated and everyone is treated with dignity and courtesy.\nAccountability: We take responsibility for our actions and decisions. We follow all relevant laws and regulations, and we strive to 

In [10]:
from langchain_community.vectorstores import Chroma

chunks_txt = text_splitter(txt_data, 200, 20)
vectordb = Chroma.from_documents(chunks_txt, huggingface_embedding())

##### Similarity Search

In [11]:
query = "email policy"
retriever = vectordb.as_retriever()
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing')]

Specify search keyword arguments like `k` to limit retrieval results,

In [12]:
retriever = vectordb.as_retriever(search_kwargs={"k": 1})
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='3.\tInternet and Email Policy')]

##### MMR Search

In [13]:
retriever = vectordb.as_retriever(search_type="mmr")
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Review of Policy: This policy will be reviewed periodically to ensure its alignment with evolving legal requirements and best practices for maintaining a healthy and safe workplace.'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='individual found to be in violation of this policy.')]

##### Similarity Score Threshold Retrieval

In [14]:
retriever = vectordb.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.4}
)
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='3.\tInternet and Email Policy'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Our Internet and Email Policy aims to promote safe, responsible usage of digital communication tools that align with our values and legal obligations. Each employee is expected to understand and'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Our Internet and Email Policy is established to guide the responsible and secure use of these essential tools within our organization. We recognize their significance in daily business operations and'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Confidentiality: Reserve email for the transmission of confidential information, trade secrets, and sensitive customer data only when encryption is applied. Exercise discretion when discussing')]

### Multi-Query Retriever

Distance-based retrieval finds documents by embedding proximity, but small wording changes can hurt results. 

MultiQueryRetriever improves this by generating multiple query variants with an LLM, retrieving for each, and merging unique results for broader, more diverse coverage.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/NCZCJ26bp3uKTa0gp8Agwg/multiquery.png" width="80%" alt="multiquery"/>

For the query “I like cats,” a distance retriever returns the closest vector match only. A multi-query retriever first generates multiple query variants with an LLM, retrieves for each, then returns the combined unique results.


In [18]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf")
pdf_data = loader.load()
pdf_data[1]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you . Its \ncore functionalities encompass:  \n1. Context -Aware Capabilities: LangChain facilitates the \ndevelopment of applications that

In [21]:
# Split
chunks_pdf = text_splitter(pdf_data, 500, 20)

# VectorDB
ids = vectordb.get().get("ids", [])

# We need to delete existing embeddings from previous documents and then store current document embeddings in.
if ids:
    vectordb.delete(ids) 

vectordb = Chroma.from_documents(documents=chunks_pdf, embedding=huggingface_embedding())

Use the `MultiQueryRetriever` function,

In [41]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

query = "What does the paper say about langchain?"

retriever = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(), llm=llm()
)

Set logging for the queries,

In [38]:
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [42]:
docs = retriever.invoke(query)
docs

[Document(metadata={'total_pages': 6, 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ioch1wsxkfqgfLLgmd-6Rw/langchain-paper.pdf', 'page': 1, 'creator': 'Microsoft Word', 'author': 'IEEE', 'page_label': '2', 'producer': 'PyPDF', 'creationdate': '2023-12-31T03:50:13+00:00'}, page_content='LangChain helps us to unlock the ability to harness the \nLLM’s immense potential in tasks such as document analysis, \nchatbot development, code analysis, and countless other \napplications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage barriers through translation, LangChain is ready to \nprovide the tools and programming support you need to do \nwithout it that it is not only difficult but also fresh for you . Its'),
 Document(metadata={'total_pages': 6, 'moddate': '2023-12-31T03:52:06+00:00', 'source': 'https://cf-courses-data.s3.us.cloud-

### Self-Querying Retriever

A Self-Querying Retriever uses an LLM to turn a natural-language query into a structured query, then applies it to a vector store for both semantic search and metadata filtering.

In [45]:
from langchain_core.documents import Document
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from lark import lark

In [54]:
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated", "rating": 2.0},
    ),
    Document(
        page_content="Three men walk into the Zone, three men walk out of the Zone",
        metadata={
            "year": 1979,
            "director": "Andrei Tarkovsky",
            "genre": "thriller",
            "rating": 9.9,
        },
    ),
]

In [55]:
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", 
        description="A 1-10 rating for the movie", 
        type="float"
    ),
]

In [56]:
vectordb = Chroma.from_documents(docs, huggingface_embedding())

In [57]:
document_content_description = "Brief summary of a movie."

retriever = SelfQueryRetriever.from_llm(
    llm(),
    vectordb,
    document_content_description,
    metadata_field_info,
)

# This example only specifies a filter
retriever.invoke("I want to watch a movie rated higher than 8.5")

[Document(metadata={'genre': 'thriller', 'director': 'Andrei Tarkovsky', 'rating': 9.9, 'year': 1979}, page_content='Three men walk into the Zone, three men walk out of the Zone'),
 Document(metadata={'genre': 'thriller', 'rating': 9.9, 'director': 'Andrei Tarkovsky', 'year': 1979}, page_content='Three men walk into the Zone, three men walk out of the Zone'),
 Document(metadata={'rating': 8.6, 'year': 2006, 'director': 'Satoshi Kon'}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea'),
 Document(metadata={'rating': 8.6, 'director': 'Satoshi Kon', 'year': 2006}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea')]

In [58]:
# This example specifies a query and a filter
retriever.invoke("Has Greta Gerwig directed any movies about women")

[Document(metadata={'rating': 8.3, 'year': 2019, 'director': 'Greta Gerwig'}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them'),
 Document(metadata={'director': 'Greta Gerwig', 'rating': 8.3, 'year': 2019}, page_content='A bunch of normal-sized women are supremely wholesome and some men pine after them')]

In [59]:
# This example specifies a composite filter
retriever.invoke("What's a highly rated (above 8.5) science fiction film?")

[Document(metadata={'rating': 8.6, 'year': 2006, 'director': 'Satoshi Kon'}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea'),
 Document(metadata={'director': 'Satoshi Kon', 'rating': 8.6, 'year': 2006}, page_content='A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea'),
 Document(metadata={'director': 'Andrei Tarkovsky', 'year': 1979, 'rating': 9.9, 'genre': 'thriller'}, page_content='Three men walk into the Zone, three men walk out of the Zone'),
 Document(metadata={'year': 1979, 'genre': 'thriller', 'rating': 9.9, 'director': 'Andrei Tarkovsky'}, page_content='Three men walk into the Zone, three men walk out of the Zone')]

### Parent Document Retriever

Document splitting has a tradeoff: small chunks improve embedding accuracy, while larger chunks preserve context. 

`ParentDocumentRetriever` balances this by indexing small chunks, then returning their larger parent documents at retrieval time.

In [60]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.storage import InMemoryStore

In [61]:
# Set two splitters. One is with big chunk size (parent) and one is with small chunk size (child)
parent_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=20, separator='\n')
child_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator='\n')

In [63]:
vectordb = Chroma(
    collection_name="split_parents", embedding_function=huggingface_embedding()
)
# The storage layer for the parent documents
store = InMemoryStore()

/var/folders/td/rtvtvvjn3x1gfqswnc7bb6hm0000gn/T/ipykernel_75660/4029313670.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


In [64]:
retriever = ParentDocumentRetriever(
    vectorstore=vectordb,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
retriever.add_documents(txt_data)

In [66]:
len(list(store.yield_keys()))

19

In [67]:
sub_docs = vectordb.similarity_search("smoking policy")
print(sub_docs[0].page_content)

5.	Smoking Policy


In [68]:
retrieved_docs = retriever.invoke("smoking policy")
print(retrieved_docs[0].page_content)

Cost Management: Keep personal phone usage separate from company accounts and reimburse the company for any personal charges on company-issued phones.
Compliance: Adhere to all pertinent laws and regulations concerning mobile phone usage, including those related to data protection and privacy.
Lost or Stolen Devices: Immediately report any lost or stolen mobile devices to the IT department or your supervisor.
Consequences: Non-compliance with this policy may lead to disciplinary actions, including the potential loss of mobile phone privileges.
The Mobile Phone Policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards. Every employee is expected to comprehend and abide by these guidelines. Regular reviews of the policy ensure its ongoing alignment with evolving technology and security best practices.
5.	Smoking Policy


### 📝 Exercise 1

Retrieve the top two results for the company policy document for the query "smoking policy" using the Vector Store-Backed Retriever.


In [69]:
# Your code here
loader = TextLoader("companypolicies.txt")
txt_data = loader.load()

chunks_txt = text_splitter(txt_data, 200, 20)
vectordb = Chroma.from_documents(chunks_txt, huggingface_embedding())

query = "smoking policy"
retriever = vectordb.as_retriever()
docs = retriever.invoke(query)
docs

[Document(metadata={'source': 'companypolicies.txt'}, page_content='5.\tSmoking Policy'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Policy Purpose: The Smoking Policy has been established to provide clear guidance and expectations concerning smoking on company premises. This policy is in place to ensure a safe and healthy'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Compliance with Applicable Laws: All employees and visitors must adhere to relevant federal, state, and local smoking laws and regulations.'),
 Document(metadata={'source': 'companypolicies.txt'}, page_content='Designated Smoking Areas: Smoking is only permitted in designated smoking areas, as marked by appropriate signage. These areas have been chosen to minimize exposure to secondhand smoke and to')]

### 📝 Exercise 2

Use the Self-Querying Retriever to invoke a query with a filter.



In [70]:
# Your code here
vectordb = Chroma.from_documents(docs, huggingface_embedding())

document_content_description = "Brief summary of a movie."

retriever = SelfQueryRetriever.from_llm(
    llm(),
    vectordb,
    document_content_description,
    metadata_field_info,
)

# This example only specifies a filter
retriever.invoke("What is the best movie based on rating?")

[Document(metadata={'year': 1979, 'director': 'Andrei Tarkovsky', 'rating': 9.9, 'genre': 'thriller'}, page_content='Three men walk into the Zone, three men walk out of the Zone'),
 Document(metadata={'genre': 'thriller', 'director': 'Andrei Tarkovsky', 'year': 1979, 'rating': 9.9}, page_content='Three men walk into the Zone, three men walk out of the Zone')]